In [2]:
from src.embeddings import get_bge_small_embed
from src.ingestion import load_documents,chunking
from src.retreival import make_query_engine
from src.Chromadb_init import build_chroma_index

In [3]:
from src.ingestion import pdfs
import time
docs=load_documents(pdfs,80)
embed=get_bge_small_embed()
ablation_configs=[
    ("384_50",384,50,"legal_bge_384"),
    ("512_100",512,100,"legal_bge_512"),
    ("768_150",768,150,"legal_bge_150")
]
indexes={}
for name,chunk_size,chunk_overlap,collection_name in ablation_configs:
    nodes=chunking(docs,chunk_size,chunk_overlap)
    idx=build_chroma_index(nodes,f"../indexes/chroma_bge_small{name}",collection_name,embed)
    t0 = time.time()
    indexes[name]=idx
    print(f"{name}: {len(nodes)} chunks | {(time.time()-t0)/60:.1f} min")

2026-03-25 16:20:11,995 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
2026-03-25 16:20:18,281 - INFO - 1 prompt is loaded, with the key: query


Total chunks created are 2886
Index creation started
384_50: 2886 chunks | 0.0 min
Total chunks created are 2269
Index creation started
512_100: 2269 chunks | 0.0 min
Total chunks created are 1625
Index creation started
768_150: 1625 chunks | 0.0 min


In [4]:
from src.reranker import get_cross_encoder_reranker

questions=[
    "what category cases mostly occurred in 1975?",
    "How many civil and criminal cases are solved? ",
    "Who is the most successful judge gave decision in minimal time?",
    "How much money laundering have happened and money value too?"
]
reranker=get_cross_encoder_reranker()

comparison=[]

for q in questions:
    row={"question":q}
    for name,idx in indexes.items():
        qe=idx.as_query_engine(
            similarity_top_k=20,
            node_postprocessors=[reranker]

        )
        t0=time.time()
        resp=qe.query(q)
        row[f"answer_{name}"]=str(resp)[:400]
        row[f"latency_{name}"]=round(time.time()-t0,2)
        comparison.append(row)
import pandas as pd
df=pd.DataFrame(comparison)
df.to_csv("../results/week2_ablation.csv",index=False)
df[["question","answer_384_50","answer_512_100","answer_768_150"]].head(3)



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-25 16:32:46,768 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-25 16:32:49,724 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-25 16:38:29,651 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-25 16:38:29,652 - INFO - Retrying request to /chat/completions in 33.000000 seconds
2026-03-25 16:39:02,830 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-25 16:39:02,831 - INFO - Retrying request to /chat/completions in 1.000000 seconds
2026-03-25 16:39:04,980 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-25 16:39:06,107 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-25 16:39:06,108 - INFO - Retrying request to /chat/completions in 40.000000 seconds
2026-03-25 16:39:46,350 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-25 16:39:46,351 - INFO - Retrying request to /chat/completions in 1.000000 seconds
2026-03-25 16:39:48,572 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-25 16:39:48,619 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-25 16:39:48,620 - INFO - Retrying request to /chat/completions in 10.000000 seconds
2026-03-25 16:39:59,355 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-25 16:39:59,458 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/co

,question,answer_384_50,answer_512_100,answer_768_150
0,what category cases mostly occurred in 1975?,"<think>\nOkay, let's tackle this query. The us...","<think>\nOkay, let's tackle this query. The us...","<think>\nOkay, let's see. The user is asking a..."
1,what category cases mostly occurred in 1975?,"<think>\nOkay, let's tackle this query. The us...","<think>\nOkay, let's tackle this query. The us...","<think>\nOkay, let's see. The user is asking a..."
2,what category cases mostly occurred in 1975?,"<think>\nOkay, let's tackle this query. The us...","<think>\nOkay, let's tackle this query. The us...","<think>\nOkay, let's see. The user is asking a..."


In [5]:
best_idx=indexes["512_100"]

#without reranker
qe_base=best_idx.as_query_engine(similarity_top_k=5)

#with reranker
qe_reranker=best_idx.as_query_engine(similarity_top_k=20,node_postprocessors=[reranker])

before_after=[]

for q in questions:
    base_resp=qe_base.query(q)
    reranked_resp=qe_reranker.query(q)
    before_after.append(
        {
            "question":q,
            "baseline_ans":str(base_resp)[:400],
            "reranked_ans":str(reranked_resp)[:400]
        }
    )

pd.DataFrame(before_after).to_csv("../results/week2_before_after.csv",index=False)


2026-03-25 17:19:25,840 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-25 17:19:28,092 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-25 17:19:29,219 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-25 17:19:30,140 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-25 17:19:30,142 - INFO - Retrying request to /chat/completions in 39.000000 seconds
2026-03-25 17:20:10,589 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-25 17:20:10,707 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-25 17:20:10,708 - INFO - Retrying request to /chat/completions in 10.000000 seconds
2026-03-25 17:20:22,979 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-25 17:20:23,901 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-25 17:20:23,902 - INFO - Retrying request to /chat/completions in 31.000000 seconds
2026-03-25 17:20:55,032 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-25 17:20:55,033 - INFO - Retrying request to /chat/completions in 5.000000 seconds
2026-03-25 17:21:01,789 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-25 17:21:01,899 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-25 17:21:01,900 - INFO - Retrying request to /chat/completions in 13.000000 seconds
2026-03-25 17:21:14,960 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-25 17:21:14,961 - INFO - Retrying request to /chat/completions in 4

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-25 17:21:20,836 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-25 17:21:20,837 - INFO - Retrying request to /chat/completions in 28.000000 seconds
2026-03-25 17:21:50,189 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
